 Part 1: Feature Engineering & Data Cleaning

---

## 📋 Project Overview

**Objective:** Develop an end-to-end algorithmic trading pipeline for a universe of N anonymized stocks.
---

### Part 1 Goals:
1. **Data Loading & Initial Exploration** - Understand the structure and quality of data
2. **Data Cleaning** - Handle missing values, outliers, and anomalies
3. **Feature Engineering** - Generate meaningful features for trading signals
4. **Feature Analysis** - Validate feature quality and relevance

---

## 📦 Step 1: Install Required Libraries

First, let's install all necessary packages. This cell is designed to work on Google Colab.

In [ ]:
# Install required packages for Colab
!pip install pandas numpy matplotlib seaborn scikit-learn scipy statsmodels ta yfinance kaggle --quiet

print("✅ All packages are ready!")

: 

## 📦 Step 2: Import Libraries

We'll use a comprehensive set of libraries for data manipulation, visualization, and technical analysis.

In [ ]:
# Core Libraries
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

# Statistical Analysis
from scipy import stats
from scipy.stats import zscore

# Technical Analysis
import ta
from ta.volatility import BollingerBands, AverageTrueRange
from ta.momentum import RSIIndicator, StochasticOscillator, ROCIndicator
from ta.trend import MACD, SMAIndicator, EMAIndicator, ADXIndicator
from ta.volume import VolumeWeightedAveragePrice, OnBalanceVolumeIndicator

# Sklearn for preprocessing
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer

print("✅ Libraries imported successfully!")
print(f"📅 Analysis Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 📥 Step 3: Setup Kaggle API & Load Data

**Instructions for Google Colab:**
1. When you run the cell below, it will prompt you to upload a file
2. Upload your `kaggle.json` file (download from kaggle.com → Settings → API)
3. The dataset will automatically download and load

**Note:** This uses the Kaggle API for reproducibility in project submissions.

In [ ]:
# === STEP 1: Setup Kaggle API ===
from google.colab import files
import os

print("📁 Please upload your kaggle.json file when prompted below:")
print("   (Get it from: kaggle.com → Account → Settings → API → Create New Token)")
print("\n" + "="*70)

uploaded = files.upload()

print("\n" + "="*70)
print("⚙️ Configuring Kaggle API...")

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("✅ Kaggle API configured successfully!\n")

# === STEP 2: Download Dataset ===
print("📥 Downloading dataset from Kaggle...")
print("   Dataset: iamspace/precog-quant-task-2026")
print("="*70)

!kaggle datasets download -d iamspace/precog-quant-task-2026
!unzip -o precog-quant-task-2026.zip

print("\n✅ Dataset downloaded and extracted!\n")

# === STEP 3: Load All Asset Files ===
print("📊 Loading all asset data files...")
print("="*70)

import glob

# Find all Asset CSV files
data_dir = 'anonymized_data'
asset_files = sorted(glob.glob(f'{data_dir}/Asset_*.csv'))

print(f"📁 Found {len(asset_files)} asset files")
print(f"   First file: {asset_files[0]}")
print(f"   Last file: {asset_files[-1]}")
print()

# Load and combine all files
all_data = []
for i, file_path in enumerate(asset_files, 1):
    try:
        df_temp = pd.read_csv(file_path)
        # Add asset identifier from filename
        asset_id = file_path.split('/')[-1].replace('.csv', '')
        df_temp['asset'] = asset_id
        all_data.append(df_temp)
        
        # Progress indicator
        if i % 20 == 0 or i == len(asset_files):
            print(f"   Loaded {i}/{len(asset_files)} files...")
    except Exception as e:
        print(f"⚠️ Error loading {file_path}: {e}")

# Combine all data into single DataFrame
df_raw = pd.concat(all_data, ignore_index=True)

print(f"\n✅ All data loaded and combined successfully!")
print(f"📊 Total Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(f"📋 Columns: {list(df_raw.columns)}")
print(f"🏢 Total Assets: {df_raw['asset'].nunique()}")

# Display date range if date column exists
date_cols = [col for col in df_raw.columns if 'date' in col.lower()]
if date_cols:
    date_col = date_cols[0]
    print(f"📅 Date Range: {df_raw[date_col].min()} to {df_raw[date_col].max()}")

print("\n" + "="*70)
print("🎉 Setup complete! You can now proceed with the analysis.")

## ✅ Step 3.1: Quick Verification

Let's verify the data loaded correctly by viewing the first few rows.

In [ ]:
# Quick data preview
print("🔍 First 5 rows of the dataset:")
print("="*80)
display(df_raw.head())

print("\n📊 Basic Statistics:")
print("="*80)
print(f"Total Records: {len(df_raw):,}")
print(f"Total Columns: {len(df_raw.columns)}")
print(f"Memory Usage: {df_raw.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print("\n✅ Data verification complete!")

## 🔍 Step 4: Initial Data Exploration

Let's understand the structure, types, and basic statistics of our data.

In [ ]:
# Display first few rows
print("📋 First 5 rows of the dataset:")
display(df_raw.head())

print("\n" + "="*80)
print("📋 Last 5 rows of the dataset:")
display(df_raw.tail())

In [ ]:
# Data types and info
print("📊 Dataset Information:")
print("="*50)
print(df_raw.info())

print("\n" + "="*50)
print("📊 Data Types:")
print(df_raw.dtypes)

In [ ]:
# Basic statistics
print("📈 Statistical Summary:")
display(df_raw.describe())

In [ ]:
# Analyze the universe of stocks
print("🏢 Universe Analysis:")
print("="*50)

# Identify ticker column (could be 'ticker', 'symbol', 'stock', etc.)
ticker_col = None
for col in df_raw.columns:
    if col.lower() in ['ticker', 'symbol', 'stock', 'asset', 'name']:
        ticker_col = col
        break

if ticker_col:
    unique_tickers = df_raw[ticker_col].nunique()
    print(f"📌 Ticker column identified: '{ticker_col}'")
    print(f"📌 Number of unique stocks: {unique_tickers}")
    print(f"\n📌 Stock list:")
    print(df_raw[ticker_col].unique())
else:
    print("⚠️ Ticker column not found. Please identify manually.")
    print(f"Available columns: {df_raw.columns.tolist()}")

In [ ]:
# Analyze date range
print("📅 Date Range Analysis:")
print("="*50)

# Identify date column
date_col = None
for col in df_raw.columns:
    if col.lower() in ['date', 'datetime', 'time', 'timestamp']:
        date_col = col
        break

if date_col:
    df_raw[date_col] = pd.to_datetime(df_raw[date_col])
    print(f"📌 Date column identified: '{date_col}'")
    print(f"📌 Start Date: {df_raw[date_col].min()}")
    print(f"📌 End Date: {df_raw[date_col].max()}")
    print(f"📌 Total trading days: {df_raw[date_col].nunique()}")
    print(f"📌 Date range: {(df_raw[date_col].max() - df_raw[date_col].min()).days} days")
else:
    print("⚠️ Date column not found. Please identify manually.")

---

## 🧹 Step 5: Data Cleaning

### 5.1 Missing Value Analysis

Missing data can significantly impact our model's performance. Let's identify and handle them appropriately.

In [ ]:
# Create a copy for cleaning
df = df_raw.copy()

# Missing value analysis
print("🔍 Missing Value Analysis:")
print("="*50)

missing_count = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Column': missing_count.index,
    'Missing Count': missing_count.values,
    'Missing Percentage': missing_percent.values
})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Percentage', ascending=False)

if len(missing_df) > 0:
    print(missing_df.to_string(index=False))
    
    # Visualize missing data
    fig, ax = plt.subplots(figsize=(12, 5))
    sns.barplot(data=missing_df, x='Column', y='Missing Percentage', palette='Reds_r', ax=ax)
    ax.set_title('Missing Values by Column', fontsize=14, fontweight='bold')
    ax.set_xlabel('Column')
    ax.set_ylabel('Missing Percentage (%)')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("✅ No missing values found!")

In [ ]:
# Handle missing values based on column type
print("🔧 Handling Missing Values:")
print("="*50)

# Identify OHLCV columns
ohlcv_cols = []
for col in df.columns:
    if col.lower() in ['open', 'high', 'low', 'close', 'volume', 'adj close', 'adjusted close', 'adj_close']:
        ohlcv_cols.append(col)

print(f"📌 Identified OHLCV columns: {ohlcv_cols}")

# Strategy for handling missing values:
# 1. For price data: Forward fill first, then backward fill
# 2. For volume: Fill with median of that stock's volume

if ticker_col and date_col:
    # Sort by ticker and date
    df = df.sort_values([ticker_col, date_col]).reset_index(drop=True)
    
    # Forward fill within each stock group
    price_cols = [col for col in ohlcv_cols if 'volume' not in col.lower()]
    volume_cols = [col for col in ohlcv_cols if 'volume' in col.lower()]
    
    for col in price_cols:
        df[col] = df.groupby(ticker_col)[col].transform(lambda x: x.ffill().bfill())
    
    for col in volume_cols:
        df[col] = df.groupby(ticker_col)[col].transform(lambda x: x.fillna(x.median()))
    
    print("✅ Missing values handled using forward/backward fill for prices and median for volume")
else:
    # Simple forward fill
    df = df.ffill().bfill()
    print("✅ Missing values handled using forward/backward fill")

# Verify
print(f"\n📊 Remaining missing values: {df.isnull().sum().sum()}")

### 5.2 Outlier Detection and Treatment

Outliers in financial data can represent:
- **Real market events** (earnings surprises, market crashes) - should be kept
- **Data errors** (wrong decimal places, incorrect entries) - should be fixed

We'll use multiple methods to identify and carefully handle outliers.

In [ ]:
def detect_outliers_zscore(data, column, threshold=3):
    """
    Detect outliers using Z-score method.
    Returns boolean mask where True indicates an outlier.
    """
    z_scores = np.abs(zscore(data[column].dropna()))
    return z_scores > threshold

def detect_outliers_iqr(data, column, k=1.5):
    """
    Detect outliers using IQR method.
    k=1.5 for mild outliers, k=3 for extreme outliers.
    """
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - k * IQR
    upper_bound = Q3 + k * IQR
    return (data[column] < lower_bound) | (data[column] > upper_bound)

# Analyze outliers in returns (more meaningful than raw prices)
print("🔍 Outlier Analysis:")
print("="*50)

# Calculate returns for each stock
close_col = None
for col in df.columns:
    if col.lower() in ['close', 'adj close', 'adjusted close', 'adj_close']:
        close_col = col
        break

if close_col and ticker_col:
    # Calculate daily returns
    df['Returns'] = df.groupby(ticker_col)[close_col].pct_change()
    
    # Detect outliers
    outliers_zscore = detect_outliers_zscore(df, 'Returns', threshold=4)
    outliers_iqr = detect_outliers_iqr(df, 'Returns', k=3)
    
    print(f"📌 Outliers detected (Z-score > 4): {outliers_zscore.sum():,}")
    print(f"📌 Outliers detected (IQR method): {outliers_iqr.sum():,}")
    
    # Visualize return distribution
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram
    axes[0].hist(df['Returns'].dropna(), bins=100, edgecolor='black', alpha=0.7)
    axes[0].set_title('Distribution of Daily Returns', fontweight='bold')
    axes[0].set_xlabel('Daily Returns')
    axes[0].set_ylabel('Frequency')
    
    # Box plot by stock (sample)
    sample_tickers = df[ticker_col].unique()[:10]
    df_sample = df[df[ticker_col].isin(sample_tickers)]
    df_sample.boxplot(column='Returns', by=ticker_col, ax=axes[1])
    axes[1].set_title('Returns Box Plot by Stock (Sample)', fontweight='bold')
    axes[1].set_xlabel('Stock')
    axes[1].set_ylabel('Daily Returns')
    plt.suptitle('')
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Could not calculate returns. Check column names.")

In [ ]:
# Handle outliers - Winsorization (clip extreme values)
# This preserves the data while limiting extreme values

def winsorize_returns(data, column, limits=(0.01, 0.99)):
    """
    Winsorize returns to limit extreme values.
    Clips values at the specified percentiles.
    """
    lower = data[column].quantile(limits[0])
    upper = data[column].quantile(limits[1])
    return data[column].clip(lower, upper)

print("🔧 Handling Outliers via Winsorization:")
print("="*50)

# Store original returns statistics
original_stats = df['Returns'].describe()

# Apply winsorization per stock
df['Returns_Clean'] = df.groupby(ticker_col)['Returns'].transform(
    lambda x: x.clip(x.quantile(0.01), x.quantile(0.99))
)

# Compare statistics
clean_stats = df['Returns_Clean'].describe()

comparison = pd.DataFrame({
    'Original': original_stats,
    'After Winsorization': clean_stats
})
print(comparison)

print("\n✅ Extreme returns clipped at 1st and 99th percentiles per stock")

### 5.3 Data Quality Checks

Let's verify OHLC relationships and detect any anomalies.

In [ ]:
# OHLC integrity checks
print("🔍 OHLC Data Integrity Checks:")
print("="*50)

# Find OHLC columns
open_col = high_col = low_col = close_col = None
for col in df.columns:
    if col.lower() == 'open':
        open_col = col
    elif col.lower() == 'high':
        high_col = col
    elif col.lower() == 'low':
        low_col = col
    elif col.lower() in ['close', 'adj close', 'adjusted close', 'adj_close']:
        close_col = col

if all([open_col, high_col, low_col, close_col]):
    # Check 1: High should be >= all other prices
    high_violations = ((df[high_col] < df[open_col]) | 
                       (df[high_col] < df[low_col]) | 
                       (df[high_col] < df[close_col])).sum()
    
    # Check 2: Low should be <= all other prices
    low_violations = ((df[low_col] > df[open_col]) | 
                      (df[low_col] > df[high_col]) | 
                      (df[low_col] > df[close_col])).sum()
    
    # Check 3: Negative prices
    negative_prices = ((df[open_col] <= 0) | (df[high_col] <= 0) | 
                       (df[low_col] <= 0) | (df[close_col] <= 0)).sum()
    
    print(f"📌 High < (Open/Low/Close) violations: {high_violations:,}")
    print(f"📌 Low > (Open/High/Close) violations: {low_violations:,}")
    print(f"📌 Negative/Zero price entries: {negative_prices:,}")
    
    if high_violations + low_violations + negative_prices == 0:
        print("\n✅ All OHLC integrity checks passed!")
    else:
        print("\n⚠️ Some data quality issues found. Fixing...")
        # Fix by recalculating high/low
        df[high_col] = df[[open_col, high_col, low_col, close_col]].max(axis=1)
        df[low_col] = df[[open_col, high_col, low_col, close_col]].min(axis=1)
        print("✅ OHLC values corrected")
else:
    print("⚠️ Could not identify all OHLC columns")

---

## 🔧 Step 6: Feature Engineering

Now comes the most crucial part - creating meaningful features that capture market dynamics. We'll create several categories of features:

1. **Price-based Features** - Returns, log returns, price ratios
2. **Momentum Indicators** - RSI, MACD, ROC, Stochastic
3. **Volatility Features** - ATR, Bollinger Bands, realized volatility
4. **Volume Features** - OBV, VWAP, volume ratios
5. **Trend Features** - Moving averages, ADX
6. **Statistical Features** - Skewness, kurtosis, z-scores

### Philosophy:
> *"The goal is not to create as many features as possible, but to create features that have predictive power while avoiding overfitting."*

In [ ]:
def create_features(data, ticker_col, date_col, open_col, high_col, low_col, close_col, volume_col):
    """
    Create comprehensive feature set for each stock.
    
    Parameters:
    -----------
    data : DataFrame
        Raw OHLCV data
    ticker_col, date_col, etc. : str
        Column names for respective fields
    
    Returns:
    --------
    DataFrame with all features
    """
    
    df_feat = data.copy()
    df_feat = df_feat.sort_values([ticker_col, date_col]).reset_index(drop=True)
    
    print("🔧 Creating features for each stock...")
    print("="*50)
    
    feature_list = []
    
    for ticker in df_feat[ticker_col].unique():
        stock_df = df_feat[df_feat[ticker_col] == ticker].copy()
        stock_df = stock_df.sort_values(date_col).reset_index(drop=True)
        
        # =====================
        # 1. PRICE-BASED FEATURES
        # =====================
        
        # Returns at different horizons
        stock_df['return_1d'] = stock_df[close_col].pct_change(1)
        stock_df['return_5d'] = stock_df[close_col].pct_change(5)
        stock_df['return_10d'] = stock_df[close_col].pct_change(10)
        stock_df['return_20d'] = stock_df[close_col].pct_change(20)
        
        # Log returns (more suitable for modeling)
        stock_df['log_return_1d'] = np.log(stock_df[close_col] / stock_df[close_col].shift(1))
        stock_df['log_return_5d'] = np.log(stock_df[close_col] / stock_df[close_col].shift(5))
        
        # Overnight gap (close to open)
        stock_df['overnight_gap'] = (stock_df[open_col] - stock_df[close_col].shift(1)) / stock_df[close_col].shift(1)
        
        # Intraday range
        stock_df['intraday_range'] = (stock_df[high_col] - stock_df[low_col]) / stock_df[close_col]
        
        # Price position within day's range
        stock_df['close_position'] = (stock_df[close_col] - stock_df[low_col]) / (stock_df[high_col] - stock_df[low_col] + 1e-10)
        
        # =====================
        # 2. MOMENTUM INDICATORS
        # =====================
        
        # RSI (Relative Strength Index) - measures overbought/oversold
        stock_df['rsi_14'] = RSIIndicator(stock_df[close_col], window=14).rsi()
        stock_df['rsi_7'] = RSIIndicator(stock_df[close_col], window=7).rsi()
        
        # MACD (Moving Average Convergence Divergence)
        macd = MACD(stock_df[close_col])
        stock_df['macd'] = macd.macd()
        stock_df['macd_signal'] = macd.macd_signal()
        stock_df['macd_diff'] = macd.macd_diff()
        
        # Rate of Change
        stock_df['roc_10'] = ROCIndicator(stock_df[close_col], window=10).roc()
        stock_df['roc_20'] = ROCIndicator(stock_df[close_col], window=20).roc()
        
        # Stochastic Oscillator
        stoch = StochasticOscillator(stock_df[high_col], stock_df[low_col], stock_df[close_col])
        stock_df['stoch_k'] = stoch.stoch()
        stock_df['stoch_d'] = stoch.stoch_signal()
        
        # =====================
        # 3. VOLATILITY FEATURES
        # =====================
        
        # Average True Range
        stock_df['atr_14'] = AverageTrueRange(stock_df[high_col], stock_df[low_col], stock_df[close_col], window=14).average_true_range()
        stock_df['atr_ratio'] = stock_df['atr_14'] / stock_df[close_col]  # Normalized ATR
        
        # Bollinger Bands
        bb = BollingerBands(stock_df[close_col], window=20, window_dev=2)
        stock_df['bb_high'] = bb.bollinger_hband()
        stock_df['bb_low'] = bb.bollinger_lband()
        stock_df['bb_mid'] = bb.bollinger_mavg()
        stock_df['bb_width'] = (stock_df['bb_high'] - stock_df['bb_low']) / stock_df['bb_mid']
        stock_df['bb_position'] = (stock_df[close_col] - stock_df['bb_low']) / (stock_df['bb_high'] - stock_df['bb_low'] + 1e-10)
        
        # Realized Volatility (rolling std of returns)
        stock_df['volatility_10d'] = stock_df['return_1d'].rolling(10).std() * np.sqrt(252)
        stock_df['volatility_20d'] = stock_df['return_1d'].rolling(20).std() * np.sqrt(252)
        stock_df['volatility_ratio'] = stock_df['volatility_10d'] / (stock_df['volatility_20d'] + 1e-10)
        
        # =====================
        # 4. VOLUME FEATURES
        # =====================
        
        # Volume ratios
        stock_df['volume_ma_10'] = stock_df[volume_col].rolling(10).mean()
        stock_df['volume_ma_20'] = stock_df[volume_col].rolling(20).mean()
        stock_df['volume_ratio'] = stock_df[volume_col] / (stock_df['volume_ma_20'] + 1)
        
        # On-Balance Volume
        stock_df['obv'] = OnBalanceVolumeIndicator(stock_df[close_col], stock_df[volume_col]).on_balance_volume()
        stock_df['obv_ma_10'] = stock_df['obv'].rolling(10).mean()
        stock_df['obv_trend'] = stock_df['obv'] / (stock_df['obv_ma_10'] + 1e-10)
        
        # Price-Volume relationship
        stock_df['pv_corr_10'] = stock_df[close_col].rolling(10).corr(stock_df[volume_col])
        
        # =====================
        # 5. TREND FEATURES
        # =====================
        
        # Simple Moving Averages
        stock_df['sma_5'] = SMAIndicator(stock_df[close_col], window=5).sma_indicator()
        stock_df['sma_10'] = SMAIndicator(stock_df[close_col], window=10).sma_indicator()
        stock_df['sma_20'] = SMAIndicator(stock_df[close_col], window=20).sma_indicator()
        stock_df['sma_50'] = SMAIndicator(stock_df[close_col], window=50).sma_indicator()
        
        # Price relative to MAs
        stock_df['price_sma_5_ratio'] = stock_df[close_col] / stock_df['sma_5']
        stock_df['price_sma_20_ratio'] = stock_df[close_col] / stock_df['sma_20']
        stock_df['price_sma_50_ratio'] = stock_df[close_col] / stock_df['sma_50']
        
        # MA crossovers
        stock_df['sma_5_20_cross'] = stock_df['sma_5'] / stock_df['sma_20']
        stock_df['sma_10_50_cross'] = stock_df['sma_10'] / stock_df['sma_50']
        
        # Exponential Moving Averages
        stock_df['ema_12'] = EMAIndicator(stock_df[close_col], window=12).ema_indicator()
        stock_df['ema_26'] = EMAIndicator(stock_df[close_col], window=26).ema_indicator()
        stock_df['ema_cross'] = stock_df['ema_12'] / stock_df['ema_26']
        
        # ADX (Average Directional Index) - trend strength
        adx = ADXIndicator(stock_df[high_col], stock_df[low_col], stock_df[close_col], window=14)
        stock_df['adx'] = adx.adx()
        stock_df['adx_pos'] = adx.adx_pos()
        stock_df['adx_neg'] = adx.adx_neg()
        
        # =====================
        # 6. STATISTICAL FEATURES
        # =====================
        
        # Rolling statistics of returns
        stock_df['return_skew_20'] = stock_df['return_1d'].rolling(20).skew()
        stock_df['return_kurt_20'] = stock_df['return_1d'].rolling(20).kurt()
        
        # Z-score of price (mean reversion signal)
        stock_df['price_zscore_20'] = (stock_df[close_col] - stock_df[close_col].rolling(20).mean()) / (stock_df[close_col].rolling(20).std() + 1e-10)
        stock_df['price_zscore_50'] = (stock_df[close_col] - stock_df[close_col].rolling(50).mean()) / (stock_df[close_col].rolling(50).std() + 1e-10)
        
        # Rolling min/max position
        stock_df['high_20d'] = stock_df[high_col].rolling(20).max()
        stock_df['low_20d'] = stock_df[low_col].rolling(20).min()
        stock_df['range_position_20d'] = (stock_df[close_col] - stock_df['low_20d']) / (stock_df['high_20d'] - stock_df['low_20d'] + 1e-10)
        
        feature_list.append(stock_df)
    
    # Combine all stocks
    df_features = pd.concat(feature_list, ignore_index=True)
    
    print(f"✅ Feature engineering complete!")
    print(f"📊 Total features created: {df_features.shape[1] - data.shape[1]}")
    
    return df_features

In [ ]:
# Find volume column
volume_col = None
for col in df.columns:
    if col.lower() == 'volume':
        volume_col = col
        break

print(f"📌 Columns identified:")
print(f"   - Ticker: {ticker_col}")
print(f"   - Date: {date_col}")
print(f"   - Open: {open_col}")
print(f"   - High: {high_col}")
print(f"   - Low: {low_col}")
print(f"   - Close: {close_col}")
print(f"   - Volume: {volume_col}")
print()

# Create features
df_features = create_features(
    df, 
    ticker_col=ticker_col,
    date_col=date_col,
    open_col=open_col,
    high_col=high_col,
    low_col=low_col,
    close_col=close_col,
    volume_col=volume_col
)

In [ ]:
# Display feature summary
print("📊 Feature Summary:")
print("="*50)
print(f"Total rows: {df_features.shape[0]:,}")
print(f"Total columns: {df_features.shape[1]}")

# List all new features by category
feature_categories = {
    'Returns': ['return_1d', 'return_5d', 'return_10d', 'return_20d', 'log_return_1d', 'log_return_5d'],
    'Price Structure': ['overnight_gap', 'intraday_range', 'close_position'],
    'Momentum': ['rsi_14', 'rsi_7', 'macd', 'macd_signal', 'macd_diff', 'roc_10', 'roc_20', 'stoch_k', 'stoch_d'],
    'Volatility': ['atr_14', 'atr_ratio', 'bb_width', 'bb_position', 'volatility_10d', 'volatility_20d', 'volatility_ratio'],
    'Volume': ['volume_ratio', 'obv_trend', 'pv_corr_10'],
    'Trend': ['price_sma_5_ratio', 'price_sma_20_ratio', 'price_sma_50_ratio', 'sma_5_20_cross', 'ema_cross', 'adx'],
    'Statistical': ['return_skew_20', 'return_kurt_20', 'price_zscore_20', 'price_zscore_50', 'range_position_20d']
}

print("\n📋 Features by Category:")
for category, features in feature_categories.items():
    print(f"\n{category}:")
    for feat in features:
        if feat in df_features.columns:
            print(f"   ✓ {feat}")

In [ ]:
# Display sample of features
print("📊 Sample of Feature Data:")
sample_cols = [ticker_col, date_col, close_col, 'return_1d', 'rsi_14', 'macd', 'volatility_20d', 'volume_ratio', 'adx']
sample_cols = [col for col in sample_cols if col in df_features.columns]
display(df_features[sample_cols].head(20))

---

## 📊 Step 7: Feature Analysis & Visualization

Let's analyze the quality and distribution of our features.

In [ ]:
# Handle NaN values in features (from rolling calculations)
print("🔧 Handling NaN values from rolling calculations:")
print("="*50)

# Check NaN counts
nan_counts = df_features.isnull().sum()
nan_cols = nan_counts[nan_counts > 0]

print(f"Columns with NaN values: {len(nan_cols)}")
print(f"Max NaN count: {nan_cols.max():,}")

# Drop rows with NaN (typically first ~50 rows per stock due to rolling windows)
initial_rows = len(df_features)
df_features = df_features.dropna()
final_rows = len(df_features)

print(f"\n📌 Rows before: {initial_rows:,}")
print(f"📌 Rows after: {final_rows:,}")
print(f"📌 Rows dropped: {initial_rows - final_rows:,} ({(initial_rows - final_rows) / initial_rows * 100:.2f}%)")

In [ ]:
# Feature distribution visualization
print("📊 Feature Distribution Analysis:")

# Select key features to visualize
viz_features = ['return_1d', 'rsi_14', 'macd', 'volatility_20d', 'volume_ratio', 'adx', 'price_zscore_20', 'bb_position']
viz_features = [f for f in viz_features if f in df_features.columns]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, feat in enumerate(viz_features):
    if i < len(axes):
        axes[i].hist(df_features[feat].dropna(), bins=50, edgecolor='black', alpha=0.7)
        axes[i].set_title(feat, fontweight='bold')
        axes[i].set_xlabel('Value')
        axes[i].set_ylabel('Frequency')

plt.suptitle('Distribution of Key Features', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Feature correlation analysis
print("📊 Feature Correlation Analysis:")

# Select numeric features for correlation
numeric_features = ['return_1d', 'return_5d', 'rsi_14', 'macd', 'volatility_20d', 
                    'volume_ratio', 'adx', 'price_zscore_20', 'bb_position', 
                    'stoch_k', 'atr_ratio', 'ema_cross']
numeric_features = [f for f in numeric_features if f in df_features.columns]

corr_matrix = df_features[numeric_features].corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0, fmt='.2f', 
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Identify highly correlated features
print("\n⚠️ Highly Correlated Feature Pairs (|r| > 0.8):")
high_corr = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > 0.8:
            high_corr.append((corr_matrix.columns[i], corr_matrix.columns[j], corr_matrix.iloc[i, j]))

for pair in high_corr:
    print(f"   {pair[0]} ↔ {pair[1]}: {pair[2]:.3f}")

In [ ]:
# Analyze feature predictive power (correlation with future returns)
print("📊 Feature Predictive Power Analysis:")
print("="*50)

# Create forward return (target variable)
df_features['forward_return_1d'] = df_features.groupby(ticker_col)['return_1d'].shift(-1)
df_features['forward_return_5d'] = df_features.groupby(ticker_col)['return_5d'].shift(-1)

# Calculate correlation of features with forward returns
feature_cols = [col for col in df_features.columns if col not in 
                [ticker_col, date_col, 'forward_return_1d', 'forward_return_5d', 
                 open_col, high_col, low_col, close_col, volume_col,
                 'Returns', 'Returns_Clean']]

correlations_1d = {}
correlations_5d = {}

for feat in feature_cols:
    try:
        corr_1d = df_features[feat].corr(df_features['forward_return_1d'])
        corr_5d = df_features[feat].corr(df_features['forward_return_5d'])
        if not np.isnan(corr_1d):
            correlations_1d[feat] = corr_1d
        if not np.isnan(corr_5d):
            correlations_5d[feat] = corr_5d
    except:
        pass

# Sort by absolute correlation
sorted_corr_1d = sorted(correlations_1d.items(), key=lambda x: abs(x[1]), reverse=True)[:15]
sorted_corr_5d = sorted(correlations_5d.items(), key=lambda x: abs(x[1]), reverse=True)[:15]

print("\n📈 Top 15 Features by Correlation with 1-Day Forward Return:")
for feat, corr in sorted_corr_1d:
    print(f"   {feat}: {corr:.4f}")

print("\n📈 Top 15 Features by Correlation with 5-Day Forward Return:")
for feat, corr in sorted_corr_5d:
    print(f"   {feat}: {corr:.4f}")

In [ ]:
# Visualize top predictive features
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1-Day Forward Return
top_10_1d = sorted_corr_1d[:10]
features_1d = [x[0] for x in top_10_1d]
corrs_1d = [x[1] for x in top_10_1d]
colors_1d = ['green' if c > 0 else 'red' for c in corrs_1d]

axes[0].barh(features_1d, corrs_1d, color=colors_1d, alpha=0.7)
axes[0].set_xlabel('Correlation')
axes[0].set_title('Top Features vs 1-Day Forward Return', fontweight='bold')
axes[0].axvline(x=0, color='black', linestyle='-', linewidth=0.5)

# 5-Day Forward Return
top_10_5d = sorted_corr_5d[:10]
features_5d = [x[0] for x in top_10_5d]
corrs_5d = [x[1] for x in top_10_5d]
colors_5d = ['green' if c > 0 else 'red' for c in corrs_5d]

axes[1].barh(features_5d, corrs_5d, color=colors_5d, alpha=0.7)
axes[1].set_xlabel('Correlation')
axes[1].set_title('Top Features vs 5-Day Forward Return', fontweight='bold')
axes[1].axvline(x=0, color='black', linestyle='-', linewidth=0.5)

plt.tight_layout()
plt.show()

---

## 💾 Step 8: Save Processed Data

Save the cleaned and feature-engineered data for use in Part 2.

In [ ]:
# Define output path
OUTPUT_PATH = 'processed_features.csv'

# Remove temporary columns
cols_to_drop = ['Returns', 'Returns_Clean']
cols_to_drop = [c for c in cols_to_drop if c in df_features.columns]
df_final = df_features.drop(columns=cols_to_drop)

# Save to CSV
df_final.to_csv(OUTPUT_PATH, index=False)

print(f"✅ Processed data saved to: {OUTPUT_PATH}")
print(f"📊 Final dataset shape: {df_final.shape[0]:,} rows × {df_final.shape[1]} columns")

# Also save feature names for reference
feature_names = [col for col in df_final.columns if col not in 
                 [ticker_col, date_col, open_col, high_col, low_col, close_col, volume_col]]

with open('feature_names.txt', 'w') as f:
    for name in feature_names:
        f.write(name + '\n')

print(f"📋 Feature names saved to: feature_names.txt")
print(f"📋 Number of features: {len(feature_names)}")

---

## 📝 Summary & Key Findings

### What We Accomplished:

1. **Data Cleaning:**
   - Handled missing values using forward/backward fill for prices and median imputation for volume
   - Detected and treated outliers using winsorization (clipping at 1st/99th percentiles)
   - Verified OHLC data integrity

2. **Feature Engineering:**
   - Created 50+ features across 6 categories:
     - Price-based features (returns, gaps, ranges)
     - Momentum indicators (RSI, MACD, ROC, Stochastic)
     - Volatility features (ATR, Bollinger Bands, realized volatility)
     - Volume features (OBV, volume ratios)
     - Trend features (moving averages, ADX)
     - Statistical features (z-scores, skewness, kurtosis)

3. **Feature Analysis:**
   - Analyzed feature distributions and correlations
   - Identified features with highest predictive power for forward returns
   - Flagged highly correlated feature pairs for potential dimensionality reduction

### Key Insights:

- Momentum features (RSI, MACD) show some predictive power for short-term returns
- Mean reversion signals (z-scores) may be useful for longer holding periods
- Volume-price relationships provide additional alpha signals

### Next Steps (Part 2):
- Build ML models to predict forward returns
- Generate trading signals from predictions
- Develop portfolio allocation strategy

---